In [39]:
import json
import time
import datetime
import os
from pathlib import Path
import shutil
from openai import OpenAI

import anthropic
import subprocess
import re

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

openai_key = os.environ["OPENAI_API_KEY"]
claude_key = os.environ["CLAUDE_API_KEY"]

client = OpenAI(
    api_key=openai_key,
)

client_claude = anthropic.Anthropic(api_key=claude_key)

In [43]:
def _load_mc_tool_schema(path: str = "./gpt_model_checking.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

import re

def _sanitize_param(s: str) -> str | None:
    if not isinstance(s, str):
        return None
    # strip inline comments
    s = s.split("//", 1)[0].split("#", 1)[0].strip()
    if not s:
        return None

    # NEW: require one of <, >, =
    m = re.match(r"^\s*([A-Za-z_][A-Za-z0-9_]*)\s*([<>=])\s*(.+?)\s*$", s)
    if not m:
        return None

    name, op, value = m.group(1), m.group(2), m.group(3).strip()

    # value allowed: int, float, identifier, or a simple quoted atom (no spaces inside)
    is_int    = re.fullmatch(r"-?\d+", value)
    is_float  = re.fullmatch(r"-?\d+\.\d+", value)
    is_ident  = re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value)
    is_quoted = (len(value) >= 2 and value[0] == value[-1] and value[0] in ("'", '"') and " " not in value[1:-1])

    if not (is_int or is_float or is_ident or is_quoted):
        return None
    if is_quoted:
        value = value[1:-1]

    # return normalized “NAME<VAL / NAME=VAL / NAME>VAL”
    return f"{name}{op}{value}"

# NEW —— call OpenAI responses API with tools to get `arguments.parameters`
def _infer_prob_parameters_from_model(model_text: str) -> list[str]:
    """
    Returns a list like ["MAXINT=5","MAXSEQ=5"] (may be empty on failure).
    """
    _MC_PROMPT_TMPL = (
        "Given the Event-B model below, analyze the "
        "suitable parameters to set up bound for model checking. "
        "Each parameter must be one of the forms 'NAME=VALUE', 'NAME<VALUE', or 'NAME>VALUE'. "
        "NAME must be a constant name or set."
        "The parameters array should be specific to the model.\n\n"
        "Model:\n{model}\n"
    )
    try:
        tool_schema = _load_mc_tool_schema("./gpt_model_checking.json")
    except Exception as e:
        print(f"[bounds] Could not load gpt_model_checking.json: {e}")
        return []

    prompt = _MC_PROMPT_TMPL.format(model=model_text)

    resp = client.responses.create(
        model="o3-mini-2025-01-31",
        reasoning={"effort": "high"},
        input=[
            {
                "role": "system",
                "content": (
                    "You are an Event-B assistant. Respond ONLY with JSON that "
                    "validates against the provided schema."
                ),
            },
            {"role": "user", "content": prompt},
        ],
        text={
            "format": tool_schema
        },
    )

    try:
        obj = json.loads(resp.output_text)
    except Exception as e:
        print(f"[bounds] Failed to parse output_text as JSON: {e}")
        return []

    params = None
    if isinstance(obj, dict):
        # Preferred (your printout): {"parameters": [...]}
        if isinstance(obj.get("parameters"), list):
            params = obj["parameters"]
        # Fallback (older shape): {"arguments": {"parameters": [...]}}
        elif isinstance(obj.get("arguments"), dict) and isinstance(obj["arguments"].get("parameters"), list):
            params = obj["arguments"]["parameters"]

    if not isinstance(params, list):
        return []

    print("params:", params)

    # strict sanitize + de-dup
    ordered, seen = [], set()
    for raw in params:
        cleaned = _sanitize_param(raw)
        if cleaned and cleaned not in seen:
            seen.add(cleaned)
            ordered.append(cleaned)

    print("cleaned params:", ordered)
    return ordered

# NEW —— convert ["A=1","B=2"] → ["-p","A","1","-p","B","2"]
def _params_to_probcli_flags(pairs: list[str]) -> list[str]:
    flags = []
    for item in pairs:
        # match NAME<VAL | NAME=VAL | NAME>VAL
        m = re.match(r"^\s*([A-Za-z_][A-Za-z0-9_]*)\s*([<>=])\s*(.+?)\s*$", item)
        if not m:
            continue
        name, op, value = m.group(1), m.group(2), m.group(3).strip()
        # inequality → pass as a scope constraint (your current pattern)
        # results in: -scope "NAME<VAL" or -scope "NAME>VAL"
        flags.extend(["-scope", f"{name}{op}{value}"])
    return flags



In [44]:
model_file = "./generated_code/ch3_mechanical_press_controller/synchronizedControlMachine.eventb"
model_text = Path(model_file).read_text(encoding="utf-8", errors="ignore")

In [45]:
bounds_pairs = _infer_prob_parameters_from_model(model_text)    # e.g., ["MAXINT=5","MAXSEQ=5"]
print(f"Inferred bounds: {bounds_pairs}")
prob_flags   = _params_to_probcli_flags(bounds_pairs)           # -> ["-p","MAXINT","5","-p","MAXSEQ","5"]
print(f"ProbCLI flags: {prob_flags}")

params: ['NumberOfButtons=2', 'NumberOfMotors=1', 'MaxDoorCloseWhenClutchDisengaged=1', 'NumberOfControllers=1', 'NumberOfClutches=1', 'NumberOfDoors=1', 'MaxClutchDisengageWhenDoorClosed=1', 'ButtonControllerWeakSync=true', 'DoorClutchSync=true', 'SafetyClutchEngagedDoorClosed=true', 'SafetyClutchEngagedMotorWorks=true', 'ControllerEquipmentStrongSync=true']
cleaned params: ['NumberOfButtons=2', 'NumberOfMotors=1', 'MaxDoorCloseWhenClutchDisengaged=1', 'NumberOfControllers=1', 'NumberOfClutches=1', 'NumberOfDoors=1', 'MaxClutchDisengageWhenDoorClosed=1', 'ButtonControllerWeakSync=true', 'DoorClutchSync=true', 'SafetyClutchEngagedDoorClosed=true', 'SafetyClutchEngagedMotorWorks=true', 'ControllerEquipmentStrongSync=true']
Inferred bounds: ['NumberOfButtons=2', 'NumberOfMotors=1', 'MaxDoorCloseWhenClutchDisengaged=1', 'NumberOfControllers=1', 'NumberOfClutches=1', 'NumberOfDoors=1', 'MaxClutchDisengageWhenDoorClosed=1', 'ButtonControllerWeakSync=true', 'DoorClutchSync=true', 'SafetyClut

In [46]:
p = Path(model_file)
if not p.exists():
    raise FileNotFoundError(f"Model file not found: {model_file}")

# Run from the directory so relative includes (.bcc/.bpo/…) work
cwd = str(p.parent)
target = p.name

cmd = ["probcli", target, "--model_check"]
if prob_flags:
    cmd.extend(prob_flags)   # <- inject our -p flags here

try:
    result = subprocess.run(
        cmd, cwd=cwd, capture_output=True, text=True, check=False
    )
except FileNotFoundError:
    raise RuntimeError("probcli not found in PATH")

out, err, rc = result.stdout, result.stderr, result.returncode

# Simple outcome heuristic
failed_markers = (
    "INVARIANT VIOLATION",
    "invariant violated",
    "deadlock",
    "Loading Specification Failed",
    "parse_error",
    "*** error occurred ***",
)
failed = any(m in out or m in err for m in failed_markers)
status = "OK" if (rc == 0 and not failed) else "FAIL"

# Optional: print for quick debugging
print("=== probcli CMD ===", " ".join(cmd), f"(cwd={cwd})")
print("=== STDOUT ===\n", out)
print("=== STDERR ===\n", err)

=== probcli CMD === probcli synchronizedControlMachine.eventb --model_check -scope NumberOfButtons=2 -scope NumberOfMotors=1 -scope MaxDoorCloseWhenClutchDisengaged=1 -scope NumberOfControllers=1 -scope NumberOfClutches=1 -scope NumberOfDoors=1 -scope MaxClutchDisengageWhenDoorClosed=1 -scope ButtonControllerWeakSync=true -scope DoorClutchSync=true -scope SafetyClutchEngagedDoorClosed=true -scope SafetyClutchEngagedMotorWorks=true -scope ControllerEquipmentStrongSync=true (cwd=generated_code/ch3_mechanical_press_controller)
=== STDOUT ===
 % unused_constants(12,[ButtonControllerWeakSync,ControllerEquipmentStrongSync,...])
% Runtime for SOLUTION for SETUP_CONSTANTS: 17 ms (walltime: 17 ms)
### Unbounded enumeration of MaxClutchDisengageWhenDoorClosed  : INTEGER : 1:sup ---> 1:3 
### call stack: 
 1: SETUP_CONSTANTS
 
% Runtime for SOLUTION for SETUP_CONSTANTS: 28 ms (walltime: 28 ms)
% Runtime for SOLUTION for SETUP_CONSTANTS: 0 ms (walltime: 0 ms)
% Runtime for SOLUTION for SETUP_CONST